In [ ]:
# Cell 1 -- Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Runtime -> Change runtime type -> T4 GPU")

props = torch.cuda.get_device_properties(0)
print("GPU        :", props.name)
print("VRAM       :", round(props.total_memory / 1e9, 1), "GB")
print("SM arch    : sm_" + str(props.major) + str(props.minor))
print("CUDA ver   :", torch.version.cuda)


In [ ]:
# Cell 2 -- Clone repo and switch to Operators branch
import os

REPO = "/content/SNNs-auf-GPUs"
if not os.path.exists(REPO):
    os.system("git clone https://github.com/Zuzu3290/SNNs-auf-GPUs.git " + REPO)

os.chdir(REPO)
os.system("git checkout Operators")
os.system("git pull origin Operators")
print("Working directory:", os.getcwd())


In [ ]:
# Cell 3 -- Dependency sanity check
# Only reinstalls and restarts if something is actually missing or broken.
# Safe to re-run when switching frameworks — no unnecessary restarts.
import sys, subprocess, importlib

REQUIRED = {
    "ninja":        None,
    "tonic":        None,
    "norse":        None,
    "spikingjelly": None,
    "snntorch":     None,
    "yaml":         "pyyaml",
}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg or mod)

# Check numpy < 2.0
numpy_bad = False
try:
    import numpy as np
    numpy_bad = int(np.__version__.split(".")[0]) >= 2
except Exception:
    numpy_bad = True

if numpy_bad:
    missing.append("numpy<2.0")

if missing:
    print("Installing missing/broken packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
    if numpy_bad:
        print("Numpy downgraded — restarting runtime...")
        import os; os.kill(os.getpid(), 9)
    print("Done.")
else:
    print("All dependencies present. No restart needed — continue to Cell 4.")

In [ ]:
# Cell 4 -- JIT-compile the kernel extension (skips if already built)
import os, sys, shutil
from torch.utils.cpp_extension import load

os.chdir("/content/SNNs-auf-GPUs")
ROOT  = "/content/SNNs-auf-GPUs"
KERN  = ROOT + "/src/crsc/kernels"
ATTRS = ROOT + "/acceleration/GPU_attributes"
BUILD = ROOT + "/.build/snn_forward"
SO    = BUILD + "/snn_forward.so"

if ROOT + "/src" not in sys.path:
    sys.path.insert(0, ROOT + "/src")

if os.path.exists(SO):
    # Kernel already compiled — just load it, no recompile
    print("Kernel .so found — loading from cache.")
    if BUILD not in sys.path:
        sys.path.insert(0, BUILD)
    import snn_forward as lif
else:
    print("Building kernel from source...")
    os.makedirs(BUILD, exist_ok=True)
    lif = load(
        name                = "snn_forward",
        build_directory     = BUILD,
        sources             = [
            KERN  + "/snn_binding.cpp",
            KERN  + "/snn_forward.cu",
            KERN  + "/lif_temporal.cu",
            KERN  + "/lif_warp_oriented.cu",
            ATTRS + "/energy_management.cu",
            ATTRS + "/memory_management.cu",
            ATTRS + "/throughput_optimiation.cu",
        ],
        extra_cflags        = ["-O3", "-DSNN_HAS_NVML=0"],
        extra_cuda_cflags   = ["-O3", "--use_fast_math", "-DSNN_HAS_NVML=0"],
        extra_include_paths = [ATTRS],
        verbose             = True,
    )

print("Functions:", [x for x in dir(lif) if not x.startswith("_")])

In [ ]:
# Cell 4b -- (Optional) Compile snn_runtime — only needed for the full training
# pipeline where dataset_cache, model_params, and kernel_workspace compete for VRAM.
# Skip this cell when testing the kernel in isolation.
import shutil
from torch.utils.cpp_extension import load

RUNTIME  = ROOT + "/acceleration/GPU_attributes"
RTBIND   = ROOT + "/src/runtime"
RT_BUILD = ROOT + "/.build/snn_runtime"
shutil.rmtree(RT_BUILD, ignore_errors=True)
os.makedirs(RT_BUILD)

rt = load(
    name                = "snn_runtime",
    build_directory     = RT_BUILD,
    sources             = [
        RTBIND  + "/runtime_binding.cpp",
        RUNTIME + "/memory_arbiter.cu",
    ],
    extra_cflags        = ["-O3", "-DSNN_HAS_NVML=0"],
    extra_cuda_cflags   = ["-O3", "--use_fast_math", "-DSNN_HAS_NVML=0"],
    extra_include_paths = [RUNTIME],
    verbose             = True,
)
print("snn_runtime :", [x for x in dir(rt) if not x.startswith("_")])
rt.MemoryArbiter(0).print_status()

In [ ]:
# Cell 5 -- Smoke test (lif is already loaded from Cell 4)
import torch

B, N, T = 4, 512, 25
inp     = torch.rand(B, N, T, device="cuda")
voltage = torch.zeros(B, N,   device="cuda")

spikes = lif.forward(inp, voltage)
print("Standard forward  shape:", tuple(spikes.shape))

spikes = lif.temporal_forward(inp, voltage)
print("Temporal forward  shape:", tuple(spikes.shape))

spikes, blocks, npt = lif.warp_oriented_forward(inp, voltage)
print("Warp-oriented     shape:", tuple(spikes.shape),
      "  blocks:", blocks, "  npt:", round(npt, 2))


In [ ]:
# Cell 6 -- Profiled forward: kernel time + energy
voltage.zero_()

result = lif.forward_profiled(inp, voltage, v_th=1.0, tau_inv=0.1)
spikes, elapsed_ms, pwr_before, pwr_after, energy_mj, nvml_ok = result

print("Profiled forward")
print("  Kernel time : {:.3f} ms".format(elapsed_ms))
print("  Spike rate  : {:.1f}%".format(spikes.mean().item() * 100))
if nvml_ok:
    print("  Power before: {:.1f} mW".format(pwr_before))
    print("  Power after : {:.1f} mW".format(pwr_after))
    print("  Energy used : {:.4f} mJ".format(energy_mj))
else:
    print("  Power/Energy: N/A (NVML not available on this GPU tier)")


In [ ]:
# Cell 7 -- Benchmark: warp-oriented kernel vs PyTorch baseline
# warp_oriented_forward is async (no CPU-GPU sync per call) -- fair comparison
import time

RUNS  = 200
WARMUP = 20

# --- Warm up GPU --------------------------------------------------
voltage.zero_()
for _ in range(WARMUP):
    lif.warp_oriented_forward(inp, voltage)
torch.cuda.synchronize()

# --- Warp-oriented kernel (async) ---------------------------------
voltage.zero_()
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(RUNS):
    lif.warp_oriented_forward(inp, voltage)
torch.cuda.synchronize()  # one sync at the end, not per call
kernel_ms = (time.perf_counter() - t0) * 1000 / RUNS

# --- PyTorch baseline (same async pattern) ------------------------
def lif_torch(x, v, v_th=1.0, tau_inv=0.1):
    v_new = v * (1.0 - tau_inv) + x
    spk   = (v_new >= v_th).float()
    return spk, v_new * (1.0 - spk)

v_pt = torch.zeros(B, N, device="cuda")
for _ in range(WARMUP):
    for t in range(T):
        _, v_pt = lif_torch(inp[:, :, t], v_pt)
torch.cuda.synchronize()

v_pt.zero_()
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(RUNS):
    for t in range(T):
        _, v_pt = lif_torch(inp[:, :, t], v_pt)
torch.cuda.synchronize()
torch_ms = (time.perf_counter() - t0) * 1000 / RUNS

# --- Timed kernel measurement (pure kernel time, no Python overhead) --
voltage.zero_()
_, elapsed_ms, blocks, npt = lif.warp_oriented_timed(inp, voltage)

print("Warp-oriented kernel (wall) : {:.3f} ms / forward".format(kernel_ms))
print("PyTorch baseline   (wall)  : {:.3f} ms / forward".format(torch_ms))
print("Speedup                    : {:.2f}x".format(torch_ms / kernel_ms))
print("")
print("Pure kernel time (GPU only): {:.4f} ms".format(elapsed_ms))
print("Blocks launched            : {}".format(blocks))
print("Neurons per thread         : {:.2f}".format(npt))


In [ ]:
# Cell 8 -- Full training run with kernel ON
# Runs inline (not subprocess) so errors and training output appear here directly.
import os, sys, yaml
from pathlib import Path

REPO = "/content/SNNs-auf-GPUs"
os.chdir(REPO)
os.system("git pull origin Operators")

# Ensure src/ and build dir are on the path for this process
for p in [REPO + "/src", REPO + "/.build/snn_forward"]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Set kernel=ON, warp_oriented, short run
cfg_path = Path(REPO + "/SNN_module.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["training"]["kernel"]               = "ON"
cfg["training"]["kernel_mode"]          = "warp_oriented"
cfg["training"]["epochs"]               = 1
cfg["training"]["iterations_per_epoch"] = 20
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False))
print("Config updated.")

# Run training directly — full traceback visible if it crashes
import torch
from skeleton import Settings
from learning.frameworks.snn_torch import SNN_TORCH
from learning.training import SNNTrainer
from event_data_workflow import NeuromorphicEncoder

cfg_obj = Settings()
device  = torch.device(cfg_obj.DEVICE)

encoder = NeuromorphicEncoder(cfg_obj)
train_loader, _ = encoder.get_dataloaders()

model   = SNN_TORCH(cfg_obj)
trainer = SNNTrainer(model, train_loader, cfg_obj, device)
results = trainer.train(checkpoint_dir="./checkpoints")

print("\nTraining complete.")
print("  Final loss     :", round(results["loss_history"][-1], 4))
print("  Final accuracy :", round(results["accuracy_history"][-1], 4))
print("  Final spike rate:", round(results["spike_rate_history"][-1], 4))